### **Accuracy of 80.19% with 7 features and a small model of 3 layers**

In [9]:
from google.colab import drive

# Monter Google Drive
drive.mount('/content/drive')

# Chemin réel du dataset
DATA_DIR = "/content/drive/MyDrive/data"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [18]:
DATA_DIR = 'data'

In [19]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm


from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [20]:
def extract_features(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (64, 64))

    features = []

    # ---------- RGB ----------
    for i in range(3):
        features.append(img[:, :, i].mean())
        features.append(img[:, :, i].std())

    # ---------- HSV ----------
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    for i in range(3):
        features.append(hsv[:, :, i].mean())
        features.append(hsv[:, :, i].std())

    # ---------- Saturation moyenne ----------
    features.append(hsv[:, :, 1].mean())

    # ---------- Nombre de couleurs différentes ----------
    pixels = img.reshape(-1, 3)
    unique_colors = np.unique(pixels, axis=0)
    num_unique_colors = len(unique_colors)
    features.append(num_unique_colors)
    features.append(num_unique_colors / (img.shape[0] * img.shape[1]))

    # ---------- Couleurs dominantes (KMeans) ----------
    kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)
    kmeans.fit(pixels)
    features.append(len(np.unique(kmeans.labels_)))

    # ---------- Contours ----------
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 100, 200)
    features.append(edges.sum() / 255)

    # ---------- Netteté ----------
    laplacian = cv2.Laplacian(gray, cv2.CV_64F)
    features.append(laplacian.var())

    return np.array(features)

In [21]:
labels_path = os.path.join(DATA_DIR, "train_labels.csv")
labels_df = pd.read_csv(labels_path)

# Encodage des labels texte (apple, google, ...) en entiers 0..6
label_encoder = LabelEncoder()
labels_df["Label_enc"] = label_encoder.fit_transform(labels_df["Label"])
class_names = label_encoder.classes_

labels_df.head()

,Id,Label,Label_enc
0,1,samsung,5
1,2,apple,0
2,3,facebook,1
3,4,facebook,1
4,5,google,2


In [22]:
X = []
y = []

for idx, row in tqdm(labels_df.iterrows(), total=len(labels_df), desc="Extracting train features"):
    img_name = f"{idx+1:05d}.png"
    img_path = os.path.join(DATA_DIR, "train", img_name)

    features = extract_features(img_path)
    X.append(features)
    # On utilise la version encodée en entier plutôt que le label texte
    y.append(row["Label_enc"])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)

print("Train features shape:", X.shape)
print("Train labels shape:", y.shape, "- dtype:", y.dtype)


Extracting train features: 100%|██████████| 9879/9879 [05:35<00:00, 29.44it/s]

Train features shape: (9879, 18)
Train labels shape: (9879,) - dtype: int64


In [23]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

In [24]:
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [26]:
model = Sequential([
    tf.keras.Input(shape=(X.shape[1],)),
    Dense(256, activation="relu"),
    Dropout(0.4),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(64, activation="relu"),
    Dense(7, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_11 (Dense)                │ (None, 256)            │         4,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 7)              │           455 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 46,471 (181.53 KB)

 Trainable params: 46,471 (181.53 KB)

 Non-trainable params: 0 (0.00 B)

In [33]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=500,
    batch_size=32,
    callbacks=[early_stopping]
)


Epoch 1/500
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8187 - loss: 0.4888 - val_accuracy: 0.8011 - val_loss: 0.5821
Epoch 2/500
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 929us/step - accuracy: 0.8245 - loss: 0.4906 - val_accuracy: 0.8062 - val_loss: 0.5831
Epoch 3/500
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 877us/step - accuracy: 0.8158 - loss: 0.5026 - val_accuracy: 0.8021 - val_loss: 0.5797
Epoch 4/500
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 865us/step - accuracy: 0.8172 - loss: 0.4895 - val_accuracy: 0.8047 - val_loss: 0.5783
Epoch 5/500
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 862us/step - accuracy: 0.8222 - loss: 0.4808 - val_accuracy: 0.8047 - val_loss: 0.5860
Epoch 6/500
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 907us/step - accuracy: 0.8142 - loss: 0.4945 - val_accuracy: 0.8036 - val_loss: 0.5786
Epoch 7/500
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 903us/step - accuracy: 0.8187 - loss: 0.4859 - val_accuracy: 0.8062 - val_loss: 0.5846
Epoch 8/500
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 908us/step - accuracy: 0.8184 - loss: 0.4

In [34]:
X_test = []

for i in tqdm(range(10001, 19880), desc="Extracting test features"):
    img_name = f"{i}.png"
    img_path = os.path.join(DATA_DIR, "test", img_name)

    features = extract_features(img_path)
    X_test.append(features)

X_test = scaler.transform(np.array(X_test))

print("Test features shape:", X_test.shape)

Extracting test features:  78%|███████▊  | 7659/9879 [04:06<01:13, 30.06it/s]/Users/simonamar-roisenberg/Downloads/Computer-Vision-Project/.venv/lib/python3.12/site-packages/sklearn/base.py:1336: ConvergenceWarning: Number of distinct clusters (1) found smaller than n_clusters (5). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
Extracting test features: 100%|██████████| 9879/9879 [05:19<00:00, 30.93it/s]

Test features shape: (9879, 18)


In [35]:
predictions = model.predict(X_test)
label_indices = np.argmax(predictions, axis=1)

# On reconvertit les entiers 0..6 en labels texte d'origine (apple, google, ...)
labels_pred = label_encoder.inverse_transform(label_indices)

submission = pd.DataFrame({
    "Id": range(10001, 19880),
    "Label": labels_pred
})

submission_path = os.path.join(DATA_DIR, "submission.csv")
submission.to_csv(submission_path, index=False)

submission.head()


309/309 ━━━━━━━━━━━━━━━━━━━━ 0s 403us/step


,Id,Label
0,10001,google
1,10002,facebook
2,10003,samsung
3,10004,whatsapp
4,10005,messenger
